# ETTh1 — Baseline Forecasting
Pipeline: Load → Clean → Feature Engineering → Split → Scale → Sliding Windows → Baselines → MLflow

**Baselines:** Persistence, Seasonal Naive (24h), Linear Ridge

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

from gridpulse.data.download_ett import download_ett
from gridpulse.preprocessing.cleaning import clean_ett
from gridpulse.features.feature_builder import build_features
from gridpulse.data.split_time_series import split_by_time
from gridpulse.preprocessing.scaling import ScalerWrapper
from gridpulse.preprocessing.windowing import create_windows
from gridpulse.models.naive import PersistenceForecaster, SeasonalNaiveForecaster
from gridpulse.models.linear import LinearForecaster
from gridpulse.training.train_forecasting import evaluate_baseline
from gridpulse.utils.paths import RAW_DIR

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
INPUT_LEN        = 96    # lookback window: 4 days of hourly data
HORIZONS         = [1, 6, 24, 48]  # multiple forecast horizons
STRIDE           = 1    # 1 = max overlap between windows
TARGET_COL       = "OT"
DATE_COL         = "date"

## 1. Load & Clean

In [ ]:
# Download ETTh1 if not present
download_ett()

ett_path = RAW_DIR / "ett" / "ETTh1.csv"
df_raw = pd.read_csv(ett_path)
df = clean_ett(df_raw)

print(f"Loaded ETTh1: {df.shape}")
df.head()

## 2. Feature Engineering
Run **before** split to avoid NaN at split boundaries.

In [ ]:
df_feat = build_features(df, target_col=TARGET_COL, date_col=DATE_COL)

print(f"After feature engineering: {df_feat.shape}")
print(f"Columns ({len(df_feat.columns)}): {list(df_feat.columns)}")

## 3. Train / Val / Test Split (chronological)

In [ ]:
train_df, val_df, test_df = split_by_time(
    df_feat, date_col=DATE_COL, train_ratio=0.6, val_ratio=0.2
)

print(f"Train : {len(train_df):>6} rows  ({train_df[DATE_COL].min()} → {train_df[DATE_COL].max()})")
print(f"Val   : {len(val_df):>6} rows  ({val_df[DATE_COL].min()} → {val_df[DATE_COL].max()})")
print(f"Test  : {len(test_df):>6} rows  ({test_df[DATE_COL].min()} → {test_df[DATE_COL].max()})")

## 4. Scaling
Fit scaler **on train only** to prevent data leakage.

> Note: `ScalerWrapper.fit_transform()` calls `transform()` internally, so the sklearn
> scaler must be fitted externally first. The wrapper is used here for save/load convenience.

In [ ]:
# Feature columns — reorder so OT is LAST
# This is required because naive models hard-code X[:, -1, -1] as the target column.
other_cols  = [c for c in df_feat.columns if c not in (DATE_COL, TARGET_COL)]
feature_cols = other_cols + [TARGET_COL]   # OT at index -1

target_col_idx = -1  # OT is last
print(f"Total features : {len(feature_cols)}")
print(f"Target column  : '{TARGET_COL}' at index {target_col_idx}")

# Fit scaler on train only
sklearn_scaler = StandardScaler()
sklearn_scaler.fit(train_df[feature_cols])

wrapper = ScalerWrapper(scaler=sklearn_scaler, columns=feature_cols)

train_scaled = wrapper.fit_transform(train_df)
val_scaled   = wrapper.fit_transform(val_df)
test_scaled  = wrapper.fit_transform(test_df)

## 5. Convert to Numpy Arrays

In [ ]:
train_arr = train_scaled[feature_cols].values
val_arr   = val_scaled[feature_cols].values
test_arr  = test_scaled[feature_cols].values

print(f"Train array : {train_arr.shape}")
print(f"Val   array : {val_arr.shape}")
print(f"Test  array : {test_arr.shape}")

## 6. Multi-Horizon Baselines → MLflow

In [ ]:
all_results = {}

for horizon in HORIZONS:
    print(f"\n{'='*60}")
    print(f"  Horizon = {horizon}h")
    print(f"{'='*60}")

    X_train_h, y_train_h = create_windows(
        train_arr, INPUT_LEN, horizon, stride=STRIDE, target_col_idx=target_col_idx
    )
    X_test_h, y_test_h = create_windows(
        test_arr, INPUT_LEN, horizon, stride=STRIDE, target_col_idx=target_col_idx
    )

    baselines = [
        PersistenceForecaster(horizon=horizon),
        SeasonalNaiveForecaster(seasonal_period=24, horizon=horizon),
        LinearForecaster(horizon=horizon),
    ]

    for model in baselines:
        model.fit(X_train_h, y_train_h)
        metrics = evaluate_baseline(
            model, X_test_h, y_test_h,
            experiment_name=f"baselines-horizon-{horizon}",
        )
        all_results[(horizon, model.name)] = metrics
        print(f"  {model.name:30s}  MAE={metrics['mae']:.4f}  RMSE={metrics['rmse']:.4f}  sMAPE={metrics['smape']:.2f}%")

## 7. Results Summary

In [ ]:
rows = []
for (horizon, model_name), metrics in all_results.items():
    rows.append({"horizon": horizon, "model": model_name, **metrics})

results_df = pd.DataFrame(rows).sort_values(["horizon", "mae"])
print(results_df.to_string(index=False))
results_df